In [ ]:
from google.colab import files
uploaded = files.upload()  # Yahan zip file select karo apne PC se

Saving hey-you-are-a-skilled-ai-2.zip to hey-you-are-a-skilled-ai-2 (1).zip


In [ ]:
import zipfile
with zipfile.ZipFile('hey-you-are-a-skilled-ai-2.zip', 'r') as z:
    z.extractall('/content/')

In [ ]:
%pip install pandas numpy scikit-learn joblib requests streamlit plotly tqdm pyarrow

In [ ]:
import requests, json

# Pehle actual file list dekho
response = requests.get("https://api.figshare.com/v2/articles/12162300")
data = response.json()
files = data.get('files', [])

print(f"Total files on Figshare: {len(files)}")
for f in files:
    print(f['name'])

Total files on Figshare: 2
data.json
metadata summary.csv


In [ ]:
import requests, json

response = requests.get("https://api.figshare.com/v2/articles/12162300")
data = response.json()
files = data.get('files', [])

# data.json ka content dekho
for f in files:
    print(f['name'], '→', f['download_url'])

data.json → https://ndownloader.figshare.com/files/22577774
metadata summary.csv → https://ndownloader.figshare.com/files/22577777


In [ ]:
# data.json download karke andar dekho
url = [f['download_url'] for f in files if f['name'] == 'data.json'][0]
r = requests.get(url)
content = r.json()
print(json.dumps(content, indent=2)[:3000])  # pehle 3000 chars

{
  "@type": "Dataset",
  "identifier": "10.1038/s41597-020-0481-z",
  "title": "Metadata record for: GaitRec, a large-scale ground reaction force dataset of healthy and impaired gait",
  "relatedPublications": [
    {
      "doi": "https://doi.org/10.1109/JBHI.2017.2785682",
      "@type": "ScholarlyArticle"
    },
    {
      "doi": "https://doi.org/10.1016/j.gaitpost.2017.07.009",
      "@type": "ScholarlyArticle"
    },
    {
      "doi": "https://doi.org/10.1016/j.gaitpost.2018.06.155",
      "@type": "ScholarlyArticle"
    },
    {
      "doi": "https://doi.org/10.1016/j.gaitpost.2019.10.021",
      "@type": "ScholarlyArticle"
    }
  ],
  "repositories": [
    {
      "name": "figshare",
      "value": "https://doi.org/10.6084/m9.figshare.c.4788012",
      "@type": "DataCatalog"
    }
  ],
  "assays": [
    {
      "measurementType": {
        "annotationValue": "gait measurement",
        "termAccession": "http://www.ebi.ac.uk/efo/EFO_0007680",
        "termSource": "EFO",
    

In [ ]:
import requests, json

# Yeh hai asli collection URL
collection_url = "https://api.figshare.com/v2/collections/4788012/articles"
response = requests.get(collection_url)
articles = response.json()

print(f"Total articles in collection: {len(articles)}")
for a in articles:
    print(a['id'], '-', a.get('title', ''))

Total articles in collection: 10
11394852 - Python_data_import
11394816 - GRF_F_AP_PRO_left
11394825 - GRF_F_V_RAW_right
11394819 - GRF_F_V_PRO_left
11394849 - Matlab_data_import
11394822 - GRF_F_V_RAW_left
11394795 - GRF_COP_ML_RAW_left
11394804 - GRF_F_V_PRO_right
11394798 - GRF_F_AP_RAW_right
11394810 - GRF_F_ML_RAW_right


In [ ]:
# Har article ki files dekho
for a in articles:
    r = requests.get(f"https://api.figshare.com/v2/articles/{a['id']}")
    data = r.json()
    print(f"\n=== Article: {data.get('title', '')} ===")
    for f in data.get('files', []):
        print(f"  {f['name']} ({f.get('size',0)//1024//1024} MB)")


=== Article: Python_data_import ===
  data_import.ipynb (0 MB)

=== Article: GRF_F_AP_PRO_left ===
  GRF_F_AP_PRO_left.csv (140 MB)

=== Article: GRF_F_V_RAW_right ===
  GRF_F_V_RAW_right.csv (241 MB)

=== Article: GRF_F_V_PRO_left ===
  GRF_F_V_PRO_left.csv (130 MB)

=== Article: Matlab_data_import ===
  data_import.m (0 MB)

=== Article: GRF_F_V_RAW_left ===
  GRF_F_V_RAW_left.csv (242 MB)

=== Article: GRF_COP_ML_RAW_left ===
  GRF_COP_ML_RAW_left.csv (269 MB)

=== Article: GRF_F_V_PRO_right ===
  GRF_F_V_PRO_right.csv (130 MB)

=== Article: GRF_F_AP_RAW_right ===
  GRF_F_AP_RAW_right.csv (248 MB)

=== Article: GRF_F_ML_RAW_right ===
  GRF_F_ML_RAW_right.csv (244 MB)


In [ ]:
import requests, json
from pathlib import Path
from tqdm import tqdm

COLLECTION_ID = "4788012"
SIGNAL_FILES = [
    "GRF_F_V_PRO_left.csv",
    "GRF_F_V_PRO_right.csv",
    "GRF_F_AP_PRO_left.csv",
    "GRF_F_AP_PRO_right.csv",
    "GRF_F_ML_PRO_left.csv",
    "GRF_F_ML_PRO_right.csv",
]
METADATA_FILE = "GRF_metadata.csv"

def download_gaitrec_fixed(data_root="/content/hey-you-are-a-skilled-ai-2/data"):
    raw_dir = Path(data_root) / "raw" / "gaitrec"
    raw_dir.mkdir(parents=True, exist_ok=True)

    # Collection se saari articles ki file list banao
    wanted = set([METADATA_FILE] + SIGNAL_FILES)
    file_index = {}  # name -> download_url

    page = 1
    while True:
        r = requests.get(
            f"https://api.figshare.com/v2/collections/{COLLECTION_ID}/articles",
            params={"page": page, "page_size": 50}
        )
        articles = r.json()
        if not articles:
            break
        for article in articles:
            r2 = requests.get(f"https://api.figshare.com/v2/articles/{article['id']}")
            data = r2.json()
            for f in data.get("files", []):
                if f["name"] in wanted:
                    file_index[f["name"]] = (f["download_url"], f.get("size", 0))
        page += 1

    print(f"Files found: {list(file_index.keys())}")

    # Download karo
    for name in wanted:
        if name not in file_index:
            print(f"⚠️  Not found: {name}")
            continue
        url, size = file_index[name]
        target = raw_dir / name
        if target.exists() and target.stat().st_size == size:
            print(f"✅ Already exists: {name}")
            continue
        print(f"⬇️  Downloading {name} ({size//1024//1024} MB)...")
        with requests.get(url, stream=True, timeout=120) as resp:
            resp.raise_for_status()
            with target.open("wb") as fh:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    if chunk:
                        fh.write(chunk)
        print(f"✅ Done: {name}")

download_gaitrec_fixed()

Files found: ['GRF_F_AP_PRO_left.csv', 'GRF_F_V_PRO_left.csv', 'GRF_F_ML_PRO_left.csv', 'GRF_F_V_PRO_right.csv', 'GRF_F_AP_PRO_right.csv', 'GRF_F_ML_PRO_right.csv', 'GRF_metadata.csv']
✅ Already exists: GRF_F_AP_PRO_right.csv
✅ Already exists: GRF_metadata.csv
✅ Already exists: GRF_F_AP_PRO_left.csv
✅ Already exists: GRF_F_ML_PRO_left.csv
✅ Already exists: GRF_F_V_PRO_right.csv
✅ Already exists: GRF_F_V_PRO_left.csv
✅ Already exists: GRF_F_ML_PRO_right.csv


In [ ]:
import sys
sys.path.insert(0, '/content/hey-you-are-a-skilled-ai-2')

from src.gait_data import build_dataset
from src.train import train_baseline

# Features build karo
build_dataset('/content/hey-you-are-a-skilled-ai-2/data')
print("✅ Dataset built!")

# Model train karo
metrics = train_baseline(
    data_root='/content/hey-you-are-a-skilled-ai-2/data',
    model_dir='/content/hey-you-are-a-skilled-ai-2/models',
    target='target_binary',
    max_trials=5000
)
print("✅ Training done!")
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"Macro F1: {metrics['macro_f1']:.3f}")

✅ Dataset built!
✅ Training done!
Accuracy: 0.940
Macro F1: 0.760


In [ ]:
!pip install streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, threading, time

# Streamlit start karo
def run_streamlit():
    subprocess.run([
        "streamlit", "run",
        "/content/hey-you-are-a-skilled-ai-2/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(4)

# Cloudflare tunnel start karo
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# URL dhundo output mein
import re
time.sleep(5)
for _ in range(20):
    line = tunnel.stderr.readline().decode()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        print("🚀 Dashboard URL:", match.group())
        break
    time.sleep(1)

cloudflared: Text file busy
🚀 Dashboard URL: https://tracy-assessing-reforms-coal.trycloudflare.com


In [ ]:
# PyTorch already Colab mein hota hai, check karo
import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cpu
GPU available: False


In [ ]:
import sys
sys.path.insert(0, '/content/hey-you-are-a-skilled-ai-2')

from src.train_cnn import train_cnn

metrics = train_cnn(
    data_root='/content/hey-you-are-a-skilled-ai-2/data',
    model_dir='/content/hey-you-are-a-skilled-ai-2/models',
    target='target_binary',
    max_trials=25000,
    epochs=12
)

print(f"✅ CNN Training Done!")
print(f"Accuracy : {metrics['accuracy']:.3f}")
print(f"Macro F1 : {metrics['macro_f1']:.3f}")
print(f"Device   : {metrics['device']}")

✅ CNN Training Done!
Accuracy : 0.933
Macro F1 : 0.799
Device   : cpu


In [ ]:
import sys
sys.path.insert(0, '/content/hey-you-are-a-skilled-ai-2')

from src.train_cnn import train_cnn

metrics = train_cnn(
    data_root='/content/hey-you-are-a-skilled-ai-2/data',
    model_dir='/content/hey-you-are-a-skilled-ai-2/models',
    target='target_binary',
    max_trials=None,
    epochs=12
)

print(f"✅ CNN Training Done!")
print(f"Accuracy : {metrics['accuracy']:.3f}")
print(f"Macro F1 : {metrics['macro_f1']:.3f}")
print(f"Device   : {metrics['device']}")

✅ CNN Training Done!
Accuracy : 0.937
Macro F1 : 0.790
Device   : cpu
